# Project 03 — E-Commerce Logistics Network Redesign

**Dataset:** Brazilian E-Commerce (Olist) — [Download from Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

Files needed in `data/olist/`:
- `olist_orders_dataset.csv`
- `olist_order_items_dataset.csv`
- `olist_sellers_dataset.csv`
- `olist_customers_dataset.csv`
- `olist_geolocation_dataset.csv`

This notebook also runs on synthetic data if files are not present.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['axes.facecolor'] = '#111'
matplotlib.rcParams['figure.facecolor'] = '#0a0a0a'
matplotlib.rcParams['text.color'] = '#f0ede8'
matplotlib.rcParams['axes.labelcolor'] = '#a09d98'
matplotlib.rcParams['xtick.color'] = '#5a5755'
matplotlib.rcParams['ytick.color'] = '#5a5755'
matplotlib.rcParams['axes.edgecolor'] = '#2a2a2a'
matplotlib.rcParams['grid.color'] = '#1e1e1e'
print('Libraries loaded.')

## Step 1 — Load Olist Data (or Synthetic)

In [ ]:
BR_STATES = ['SP','RJ','MG','RS','PR','SC','BA','GO','DF','PE','CE','AM','ES','MT','MS','RO','MA','PA','RN','PB','AL','PI','TO','SE','AC','AP','RR']

try:
    orders = pd.read_csv('data/olist/olist_orders_dataset.csv')
    items = pd.read_csv('data/olist/olist_order_items_dataset.csv')
    sellers = pd.read_csv('data/olist/olist_sellers_dataset.csv')
    customers = pd.read_csv('data/olist/olist_customers_dataset.csv')
    print(f'Loaded real Olist data: {len(orders)} orders')
    # Merge
    df = orders.merge(items[['order_id','seller_id','price']], on='order_id')
    df = df.merge(sellers[['seller_id','seller_state']], on='seller_id')
    df = df.merge(customers[['customer_id','customer_state']], on='customer_id')
    df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
    df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
    df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])
    delivered = df[df['order_status'] == 'delivered'].copy()
    delivered['actual_days'] = (delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp']).dt.days
    delivered['estimated_days'] = (delivered['order_estimated_delivery_date'] - delivered['order_purchase_timestamp']).dt.days
    delivered['delay_days'] = delivered['actual_days'] - delivered['estimated_days']
    delivered['is_late'] = delivered['delay_days'] > 0
except FileNotFoundError:
    print('Olist data not found — generating synthetic data...')
    np.random.seed(77)
    n = 50000
    # High-delay corridors are SE sellers to N/NE customers
    se_states = ['SP','RJ','MG','ES']
    n_states = ['AM','PA','RO','AC','AP','RR','TO','MA','PI','CE','RN','PB','PE','AL','SE','BA']
    other_states = [s for s in BR_STATES if s not in se_states + n_states]
    seller_states = np.random.choice(se_states + other_states, n, p=[0.04]*len(se_states) + [0.02]*len(other_states))
    customer_states = np.random.choice(n_states + other_states, n, p=[0.04]*len(n_states) + [0.015]*len(other_states))
    # Base delay
    base_delay = np.random.normal(-1, 3, n)
    # SE->N corridor gets extra delay
    is_bad_corridor = np.array([s in se_states and c in n_states for s, c in zip(seller_states, customer_states)])
    base_delay[is_bad_corridor] += np.random.normal(6, 2, is_bad_corridor.sum())
    delivered = pd.DataFrame({
        'order_id': [f'ORD{i:06d}' for i in range(n)],
        'seller_state': seller_states,
        'customer_state': customer_states,
        'delay_days': base_delay.round(1),
        'actual_days': np.clip(np.random.normal(10, 4, n) + base_delay, 1, 45).round(0),
        'price': np.random.exponential(80, n).round(2),
        'seller_processing_days': np.clip(np.random.normal(2.5, 1.2, n), 0.5, 8),
        'carrier_transit_days': np.clip(np.random.normal(6, 2, n) + is_bad_corridor * 3, 1, 20),
        'lastmile_days': np.clip(np.random.normal(1.5, 0.8, n), 0.5, 5),
    })
    delivered['is_late'] = delivered['delay_days'] > 0
    print(f'Synthetic dataset: {len(delivered)} orders')

late_rate = delivered['is_late'].mean() * 100
print(f'\nOverall late delivery rate: {late_rate:.1f}%')

## Step 2 — OTIF Analysis & Pareto of Delay Corridors

In [ ]:
# Corridor analysis: seller_state → customer_state
corridor = delivered.groupby(['seller_state','customer_state']).agg(
    order_count=('order_id','count'),
    late_count=('is_late','sum'),
    avg_delay=('delay_days','mean')
).reset_index()
corridor['late_rate'] = corridor['late_count'] / corridor['order_count']
corridor['corridor'] = corridor['seller_state'] + ' → ' + corridor['customer_state']

# Pareto: what % of corridors cause what % of delays?
corridor_sorted = corridor.sort_values('late_count', ascending=False)
corridor_sorted['cumulative_late_pct'] = corridor_sorted['late_count'].cumsum() / corridor_sorted['late_count'].sum() * 100
corridor_sorted['pct_corridors'] = np.arange(1, len(corridor_sorted)+1) / len(corridor_sorted) * 100

p20 = corridor_sorted[corridor_sorted['pct_corridors'] <= 20].iloc[-1]
print(f'Pareto finding: Top 20% of corridors cause {p20["cumulative_late_pct"]:.1f}% of late deliveries')
print(f'\nTop 10 worst corridors by late count:')
print(corridor_sorted[['corridor','order_count','late_count','late_rate','avg_delay']].head(10).round(2).to_string(index=False))

# Pareto chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(corridor_sorted['pct_corridors'], corridor_sorted['cumulative_late_pct'],
             color='#c8f060', linewidth=2)
axes[0].axvline(20, color='#f07060', linestyle='--', linewidth=1.5, label='Top 20% corridors')
axes[0].axhline(p20['cumulative_late_pct'], color='#f07060', linestyle=':', linewidth=1)
axes[0].set_xlabel('% of Corridors')
axes[0].set_ylabel('% of Total Late Deliveries')
axes[0].set_title('Pareto: Corridors vs Late Delivery Share', color='#f0ede8', fontsize=12)
axes[0].legend()

top10 = corridor_sorted.head(10)
axes[1].barh(top10['corridor'], top10['late_rate'] * 100, color='#f07060', alpha=0.85)
axes[1].set_xlabel('Late Rate (%)')
axes[1].set_title('Top 10 Corridors by Late Rate', color='#f0ede8', fontsize=12)
plt.tight_layout()
plt.savefig('corridor_pareto.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3 — Root Cause Decomposition

In [ ]:
# Decompose total delay into 3 components
components = ['seller_processing_days','carrier_transit_days','lastmile_days']

# For late vs on-time orders
comp_summary = delivered.groupby('is_late')[components].mean()
comp_summary.index = ['On Time', 'Late']
print('Average days per component:')
print(comp_summary.round(2))

# Variance contribution analysis
total_var = delivered[components].var().sum()
var_share = (delivered[components].var() / total_var * 100).round(1)
print(f'\nVariance contribution by component:')
for c, v in var_share.items():
    print(f'  {c}: {v:.1f}%')

top_driver = var_share.idxmax()
print(f'\nPrimary root cause: {top_driver} explains {var_share.max():.1f}% of delivery variance')

# Stacked bar chart
fig, ax = plt.subplots(figsize=(10, 5))
comp_summary.plot(kind='bar', stacked=True, ax=ax,
                  color=['#c8f060','#60a8f0','#f0b860'], edgecolor='#0a0a0a', linewidth=0.5)
ax.set_title('Avg Delivery Days by Component: On Time vs Late', color='#f0ede8', fontsize=12)
ax.set_xlabel('')
ax.set_xticklabels(['On Time', 'Late'], rotation=0)
ax.legend(labels=['Seller Processing','Carrier Transit','Last Mile'])
plt.tight_layout()
plt.savefig('root_cause_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

# Export corridor data for Tableau map
corridor_sorted.to_csv('corridor_delay_analysis.csv', index=False)
print('\nCorridor data exported to corridor_delay_analysis.csv')
print('\n=== PROJECT 03 COMPLETE ===')
print(f'Late rate: {late_rate:.1f}% | Top root cause: {top_driver} ({var_share.max():.1f}% variance)')